### And now - Week 3 Day 3

## AutoGen Core

Something a little different.

This is agnostic to the underlying Agent framework

You can use AutoGen AgentChat, or you can use something else; it's an Agent interaction framework.

From that point of view, it's positioned similarly to LangGraph.

### The fundamental principle

Autogen Core decouples an agent's logic from how messages are delivered.  
The framework provides a communication infrastructure, along with agent lifecycle, and the agents are responsible for their own work.

The communication infrastructure is called a Runtime.

There are 2 types: **Standalone** and **Distributed**.

Today we will use a standalone runtime: the **SingleThreadedAgentRuntime**, a local embedded agent runtime implementation.

Tomorrow we'll briefly look at a Distributed runtime.


In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)


True

### First we define our Message object

Whatever structure we want for messages in our Agent framework.

In [2]:
@dataclass
class Message:
    content: str

### Now we define our Agent

A subclass of RoutedAgent.

Every Agent has an **Agent ID** which has 2 components:  
`agent.id.type` describes the kind of agent it is  
`agent.id.key` gives it its unique identifier

Any method with the `@message_handler` decorated will have the opportunity to receive messages.


In [4]:
class DontCareAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("DontCare")

    @message_handler
    async def on_my_message(self, message: Message, context: MessageContext) -> Message:
        return Message(content=f"This is {self.id.type}-{self.id.key}. You said '{message.content}' and I don't care what you said.")
        

### OK let's create a Standalone runtime and register our agent type

In [6]:
runtime = SingleThreadedAgentRuntime()
await DontCareAgent.register(runtime, "dont_care_agent", lambda: DontCareAgent())

AgentType(type='dont_care_agent')

### Alright! Let's start a runtime and send a message

In [7]:
runtime.start()

In [8]:
agent_id = AgentId("dont_care_agent", "default")
response = await runtime.send_message(Message(content="We are Charlie Kirk and we carry the flame"), agent_id)
print(">>> Response from agent: ", response.content)

>>> Response from agent:  This is dont_care_agent-default. You said 'We are Charlie Kirk and we carry the flame' and I don't care what you said.


In [9]:
await runtime.stop()
await runtime.close()

### OK Now let's do something more interesting

We'll use an AgentChat Assistant!

In [29]:
class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name="LLMAgent", model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, context: MessageContext) -> Message:
        print(f"{self.id.type}-{self.id.key} received message: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], context.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type}-{self.id.key} replying with: {reply}")
        return Message(content=reply)


In [30]:
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()
await MyLLMAgent.register(runtime, "MY_LLM_AGENT", lambda: MyLLMAgent())
await DontCareAgent.register(runtime, "DONT_CARE_AGENT", lambda: DontCareAgent())

AgentType(type='DONT_CARE_AGENT')

In [35]:
runtime.start() # start processing the messages in the background
response = await runtime.send_message(Message(content="Hi there!"), AgentId("MY_LLM_AGENT", "default"))
print(">>> Response from MY_LLM_AGENT: ", response.content)
response = await runtime.send_message(Message(content=response.content), AgentId("DONT_CARE_AGENT", "default"))
print(">>> Response from DONT_CARE_AGENT: ", response.content)
response = await runtime.send_message(Message(content=response.content), AgentId("MY_LLM_AGENT", "default"))
print(">>> Response from MY_LLM_AGENT: ", response.content)

MY_LLM_AGENT-default received message: Hi there!
MY_LLM_AGENT-default replying with: Hello! How can I assist you today?
>>> Response from MY_LLM_AGENT:  Hello! How can I assist you today?
>>> Response from DONT_CARE_AGENT:  This is DONT_CARE_AGENT-default. You said 'Hello! How can I assist you today?' and I don't care what you said.
MY_LLM_AGENT-default received message: This is DONT_CARE_AGENT-default. You said 'Hello! How can I assist you today?' and I don't care what you said.
MY_LLM_AGENT-default replying with: Understood! If there's something specific you'd like to talk about or ask, feel free to let me know!
>>> Response from MY_LLM_AGENT:  Understood! If there's something specific you'd like to talk about or ask, feel free to let me know!


In [36]:
await runtime.stop()
await runtime.close()

### OK now let's show this at work - let's have 3 agents interact!

In [37]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name=name, model_client=model_client)
    
    @message_handler
    async def handle_message(self, message: Message, context: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], context.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OllamaChatCompletionClient(model="mistral")
        self._delegate = AssistantAgent(name=name, model_client=model_client)

    @message_handler
    async def handle_message(self, message: Message, context: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], context.cancellation_token)
        return Message(content=response.chat_message.content)

In [44]:
JUDGE = "You are judging a game of rock, paper, scissors. The players have made their choices:\n"

class RockPaperScissorAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name=name, model_client=model_client)

    @message_handler
    async def handle_message(self, message: Message, context: MessageContext) -> Message:
        instructions = "You are playing rock, paper, scissors. Respond only with one word, one of the following: rock, paper, scissors. Do not include any other text in your response."
        message = Message(content=instructions)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)
        response2 = await self.send_message(message, inner_2)
        result = f"Player 1: {response1.content}\n Player 2: {response2.content}\n"
        judgement = f"{JUDGE}{result} Who wins?"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], context.cancellation_token)
        return Message(content=result + response.chat_message.content)

In [45]:
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RockPaperScissorAgent.register(runtime, "rock_paper_scissors", lambda: RockPaperScissorAgent("rock_paper_scissors"))
runtime.start()

In [46]:
agent_id = AgentId("rock_paper_scissors", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

Player 1: scissors
 Player 2:  rock
Player 2 wins because rock beats scissors. TERMINATE


In [47]:
await runtime.stop()
await runtime.close()